# 05 — Predictions Verification

Inspect the `predictions` collection that the API writes on each `/predict` call:
1. Total & per-symbol counts
2. Latest predictions
3. Duplicate check (unique index on `(symbol, target_kline_start_time)`)
4. Predicted vs actual — join with `closed_candles` and compute the error
5. Error summary (MAE / RMSE)

In [2]:
import sys, os
sys.path.append(os.path.abspath('..'))  # so `src` is importable from notebooks/

import pandas as pd
from src.storage.mongo import get_collection

preds = get_collection('predictions')     # forecasts written by /predict
closed = get_collection('closed_candles')  # actual completed candles

## 1. Total & per-symbol counts

In [3]:
print('Total predictions:', preds.count_documents({}))

pd.DataFrame(list(preds.aggregate([
    {'$group': {
        '_id': '$symbol',
        'count': {'$sum': 1},
        'first': {'$min': '$target_kline_start_time'},
        'last':  {'$max': '$target_kline_start_time'},
    }},
])))

Total predictions: 55


,_id,count,first,last
0,BTCUSDT,55,2026-06-28 05:10:00,2026-06-28 07:49:00


## 2. Latest 5 predictions
Each doc records the candle it was made FROM (`source_kline_start_time`) and the candle it predicts (`target_kline_start_time`).

In [3]:
docs = list(preds.find().sort('target_kline_start_time', -1).limit(5))
pd.DataFrame(docs)

,_id,symbol,target_kline_start_time,predicted_close,source_kline_start_time
0,6a40b26d74cac371e05a725d,BTCUSDT,2026-06-28 05:34:00,59907.922747,2026-06-28 05:33:00
1,6a40b23174cac371e05a71f1,BTCUSDT,2026-06-28 05:33:00,59882.052253,2026-06-28 05:32:00
2,6a40b1f574cac371e05a7185,BTCUSDT,2026-06-28 05:32:00,59870.579414,2026-06-28 05:31:00
3,6a40b19474cac371e05a7088,BTCUSDT,2026-06-28 05:31:00,59836.205601,2026-06-28 05:30:00
4,6a40b15874cac371e05a6f34,BTCUSDT,2026-06-28 05:30:00,59836.353420,2026-06-28 05:29:00


## 3. Duplicate check
The unique index on `(symbol, target_kline_start_time)` should make this empty — one stored forecast per predicted candle.

In [4]:
dups = list(preds.aggregate([
    {'$group': {
        '_id': {'symbol': '$symbol', 'target': '$target_kline_start_time'},
        'count': {'$sum': 1},
    }},
    {'$match': {'count': {'$gt': 1}}},
]))

print(f'{len(dups)} duplicate (symbol, target_kline_start_time) keys found')
dups

0 duplicate (symbol, target_kline_start_time) keys found


[]

## 4. Predicted vs actual
Join each prediction's `target_kline_start_time` to the actual close in `closed_candles` (the same logic as the `/predictions` API endpoint), then compute the error. The newest predictions have no `actual_close` yet — their candle hasn't closed.

In [5]:
SYMBOL = 'BTCUSDT'

pred_docs = list(preds.find({'symbol': SYMBOL}).sort('target_kline_start_time', 1))
df = pd.DataFrame(pred_docs)

if df.empty:
    print('No predictions yet — open the dashboard / call /predict first.')
else:
    df = df[['target_kline_start_time', 'predicted_close']].copy()

    # look up the actual close for each predicted candle in one query
    targets = df['target_kline_start_time'].tolist()
    actuals = {
        c['kline_start_time']: c['close']
        for c in closed.find(
            {'symbol': SYMBOL, 'kline_start_time': {'$in': targets}},
            {'_id': 0, 'kline_start_time': 1, 'close': 1},
        )
    }
    df['actual_close'] = df['target_kline_start_time'].map(actuals)
    df['error'] = df['predicted_close'] - df['actual_close']

df

,target_kline_start_time,predicted_close,actual_close,error
0,2026-06-28 05:10:00,60021.371941,NaN,NaN
1,2026-06-28 05:22:00,60000.777745,59969.59,31.187745
2,2026-06-28 05:25:00,59912.602912,59831.22,81.382912
3,2026-06-28 05:26:00,59832.446175,59833.24,-0.793825
4,2026-06-28 05:27:00,59832.945116,59806.78,26.165116
5,2026-06-28 05:28:00,59806.987243,59858.00,-51.012757
6,2026-06-28 05:29:00,59857.733244,59836.09,21.643244
7,2026-06-28 05:30:00,59836.353420,59835.99,0.363420
8,2026-06-28 05:31:00,59836.205601,59870.75,-34.544399
9,2026-06-28 05:32:00,59870.579414,59881.99,-11.410586


## 5. Error summary

In [6]:
if 'error' in df.columns:
    paired = df.dropna(subset=['actual_close'])
    print(f'{len(df)} predictions, {len(paired)} matched with an actual close')
    if not paired.empty:
        mae = paired['error'].abs().mean()
        rmse = (paired['error'] ** 2).mean() ** 0.5
        print(f'MAE  : {mae:,.2f}')
        print(f'RMSE : {rmse:,.2f}')
    display(paired.tail(10))
else:
    print('No predictions to summarise yet.')

13 predictions, 11 matched with an actual close
MAE  : 27.68
RMSE : 35.36


,target_kline_start_time,predicted_close,actual_close,error
2,2026-06-28 05:25:00,59912.602912,59831.22,81.382912
3,2026-06-28 05:26:00,59832.446175,59833.24,-0.793825
4,2026-06-28 05:27:00,59832.945116,59806.78,26.165116
5,2026-06-28 05:28:00,59806.987243,59858.00,-51.012757
6,2026-06-28 05:29:00,59857.733244,59836.09,21.643244
7,2026-06-28 05:30:00,59836.353420,59835.99,0.363420
8,2026-06-28 05:31:00,59836.205601,59870.75,-34.544399
9,2026-06-28 05:32:00,59870.579414,59881.99,-11.410586
10,2026-06-28 05:33:00,59882.052253,59907.99,-25.937747
11,2026-06-28 05:34:00,59907.922747,59927.98,-20.057253
